In [0]:
%pip install -U -qqqq mlflow-skinny langchain==0.2.16 langgraph-checkpoint==1.0.12 langchain_core langchain-community==0.2.16 langgraph==0.2.16 pydantic databricks_langchain==0.1.1 
dbutils.library.restartPython()

In [0]:
%run ./00_config

## Define the chat model and tools
Create a LangChain chat model that supports [LangGraph tool](https://langchain-ai.github.io/langgraph/how-tos/tool-calling/) calling.

Modify the tools your agent has access to by modifying the `uc_functions` list in [config.yml]($./config.yml). Any non-UC function spec tools can be defined in this notebook. See [LangChain - How to create tools](https://python.langchain.com/v0.2/docs/how_to/custom_tools/) and [LangChain - Using built-in tools](https://python.langchain.com/v0.2/docs/how_to/tools_builtin/).

 **_NOTE:_**  This notebook uses LangChain, however AI Agent Framework is compatible with other agent frameworks like Pyfunc and LlamaIndex.

In [0]:
import mlflow
from mlflow.models import ModelConfig

mlflow.langchain.autolog()

In [0]:

my_config = {
  "agent_prompt": """You are a highly knowledgeable and creative film assistant, specialising in analysing movie plots and helping users discover movies that match their interests or needs. You have access to a vector search retriever tool, which allows you to retrieve relevant movies based on their plot descriptions. You also have metadata on the films, such as film name, year, and category.

  Your responsibilities include:
  1. Interpreting the user's request to understand the thematic or plot-based details they are searching for.
  2. Using the retriever tool to find movies with plots that match the user's description. Retrieve the top results and summarise them clearly for the user.
  3. Providing thoughtful, concise, and contextually relevant responses to enhance the user’s understanding or interest in the films.
  4. If needed, provide additional information or insights into the films retrieved, such as key themes, genres, or cultural significance.

  When responding:
  - Always include the name of the film prominently at the start of your response, followed by a summary of its plot or other relevant details.
  - Focus on clarity and relevance to the user's query.
  - Explain why the retrieved movies fit the query when necessary.
  - Offer additional suggestions or insights to make your response engaging and helpful.

  If the user's request is ambiguous or lacks detail, ask clarifying questions before using the retriever. If the user's request is about a topic besides films, answer that you can only help with films related queries
  """,
  "temperature": 0,
  "llm_endpoint": "databricks-meta-llama-3-3-70b-instruct"
}

In [0]:
model_config = ModelConfig(development_config=my_config)

In [0]:
from langchain_community.chat_models import ChatDatabricks
from langchain_community.tools.databricks import UCFunctionToolkit

# Create the llm
llm = ChatDatabricks(endpoint=model_config.get("llm_endpoint"))
# VECTOR_INDEX_NAME=f"{UC_CATALOG}.{UC_SCHEMA}.wikipedia_vector_index"

In [0]:
# VECTOR_INDEX_NAME

In [0]:
VECTOR_SEARCH_ENDPOINT_NAME

In [0]:
from langchain.tools.retriever import create_retriever_tool
from databricks_langchain.vectorstores import DatabricksVectorSearch

# Connect to an existing Databricks Vector Search endpoint and index
vector_store = DatabricksVectorSearch(
  endpoint="one-env-shared-endpoint-5",
  # endpoint=VECTOR_SEARCH_ENDPOINT_NAME,
  index_name='hannamoazam_catalog.cookbook.wikipedia_vector_index',
  # index_name=VECTOR_INDEX_NAME,
  columns=[
    "id",    
    "Release_Year",
    "Title",
    "origin",
    "Director",
    "Cast",
    "Genre",
    "Wiki_Page",
    "Plot",
    "chunks"
  ] # TODO: Fill in with column names
).as_retriever(search_kwargs={"k": 5})

# Create a tool object that performs retrieval against our vector search index
retriever_tool = create_retriever_tool(
  vector_store,
  name="films_retriever", # TODO: Fill in your retriever's name
  description="Find films based on the description of the plot", # TODO: Fill in your retriever's description to help the LLM choose this tool
)

# Specify the return type schema of our retriever, so that evaluation and UIs can
# automatically display retrieved chunks
mlflow.models.set_retriever_schema(
    primary_key="id",
    text_column="chunks",
    doc_uri="Wiki_Page",
    name="films_retriever",
)

tools = [retriever_tool] # TODO: Remove if you have tools defined in uc and loaded in the cell above


## Output parsers
Databricks interfaces, such as the AI Playground, can optionally display pretty-printed tool calls.

Use the following helper functions to parse the LLM's output into the expected format.

(Don't worry about this part too much)

In [0]:
from typing import Iterator, Dict, Any
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    ToolMessage,
    MessageLikeRepresentation,
)

import json

def stringify_tool_call(tool_call: Dict[str, Any]) -> str:
    """
    Convert a raw tool call into a formatted string that the playground UI expects if there is enough information in the tool_call
    """
    try:
        request = json.dumps(
            {
                "id": tool_call.get("id"),
                "name": tool_call.get("name"),
                "arguments": json.dumps(tool_call.get("args", {})),
            },
            indent=2,
        )
        return f"<tool_call>{request}</tool_call>"
    except:
        return str(tool_call)


def stringify_tool_result(tool_msg: ToolMessage) -> str:
    """
    Convert a ToolMessage into a formatted string that the playground UI expects if there is enough information in the ToolMessage
    """
    try:
        result = json.dumps(
            {"id": tool_msg.tool_call_id, "content": tool_msg.content}, indent=2
        )
        return f"<tool_call_result>{result}</tool_call_result>"
    except:
        return str(tool_msg)


def parse_message(msg) -> str:
    """Parse different message types into their string representations"""
    # tool call result
    if isinstance(msg, ToolMessage):
        return stringify_tool_result(msg)
    # tool call
    elif isinstance(msg, AIMessage) and msg.tool_calls:
        tool_call_results = [stringify_tool_call(call) for call in msg.tool_calls]
        return "".join(tool_call_results)
    # normal HumanMessage or AIMessage (reasoning or final answer)
    elif isinstance(msg, (AIMessage, HumanMessage)):
        return msg.content
    else:
        print(f"Unexpected message type: {type(msg)}")
        return str(msg)


def wrap_output(stream: Iterator[MessageLikeRepresentation]) -> Iterator[str]:
    """
    Process and yield formatted outputs from the message stream.
    The invoke and stream langchain functions produce different output formats.
    This function handles both cases.
    """
    for event in stream:
        # the agent was called with invoke()
        if "messages" in event:
            for msg in event["messages"]:
                yield parse_message(msg) + "\n\n"
        # the agent was called with stream()
        else:
            for node in event:
                for key, messages in event[node].items():
                    if isinstance(messages, list):
                        for msg in messages:
                            yield parse_message(msg) + "\n\n"
                    else:
                        print("Unexpected value {messages} for key {key}. Expected a list of `MessageLikeRepresentation`'s")
                        yield str(messages)

## Create the agent
Here we provide a simple graph that uses the model and tools defined by [config.yml]($./config.yml). This graph is adapated from [this LangGraph guide](https://langchain-ai.github.io/langgraph/how-tos/react-agent-from-scratch/).


To further customize your LangGraph agent, you can refer to:
* [LangGraph - Quick Start](https://langchain-ai.github.io/langgraph/tutorials/introduction/) for explanations of the concepts used in this LangGraph agent
* [LangGraph - How-to Guides](https://langchain-ai.github.io/langgraph/how-tos/) to expand the functionality of your agent

In [0]:
from typing import (
    Annotated,
    Optional,
    Sequence,
    TypedDict,
    Union,
)

from langchain_core.language_models import LanguageModelLike
from langchain_core.messages import (
    BaseMessage,
    SystemMessage,
)
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool

from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_executor import ToolExecutor
from langgraph.prebuilt.tool_node import ToolNode


# We create the AgentState that we will pass around
# This simply involves a list of messages
class AgentState(TypedDict):
    """The state of the agent."""

    messages: Annotated[Sequence[BaseMessage], add_messages]


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolExecutor, Sequence[BaseTool]],
    agent_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    # Define the function that determines which node to go to
    def should_continue(state: AgentState):
        messages = state["messages"]
        last_message = messages[-1]
        # If there is no function call, then we finish
        if not last_message.tool_calls:
            return "end"
        else:
            return "continue"

    if agent_prompt:
        system_message = SystemMessage(content=agent_prompt)
        preprocessor = RunnableLambda(
            lambda state: [system_message] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    # Define the function that calls the model
    def call_model(
        state: AgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    workflow = StateGraph(AgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        # First, we define the start node. We use agent.
        # This means these are the edges taken after the agent node is called.
        "agent",
        # Next, we pass in the function that will determine which node is called next.
        should_continue,
        # The mapping below will be used to determine which node to go to
        {
            # If tools, then we call the tool node.
            "continue": "tools",
            # END is a special node marking that the graph should finish.
            "end": END,
        },
    )
    # We now add a unconditional edge from tools to agent.
    workflow.add_edge("tools", "agent")

    return workflow.compile()

In [0]:
from langchain_core.runnables import RunnableGenerator
from mlflow.langchain.output_parsers import ChatCompletionsOutputParser

# Create the agent with the system message if it exists
try:
    # Attempt to retrieve the agent prompt from the model configuration
    agent_prompt = model_config.get("agent_prompt")
    # Create the agent with the provided LLM, tools, and agent prompt
    agent_with_raw_output = create_tool_calling_agent(
        llm, tools, agent_prompt=agent_prompt
    )
except KeyError:
    # If no agent prompt is found, create the agent without it
    agent_with_raw_output = create_tool_calling_agent(llm, tools)
    
# Chain the agent with a RunnableGenerator to wrap the output and a ChatCompletionsOutputParser to parse the output
agent = agent_with_raw_output | RunnableGenerator(wrap_output) | ChatCompletionsOutputParser()

## Test the agent

Interact with the agent to test its output. Since this notebook called `mlflow.langchain.autolog()` you can view the trace for each step the agent takes.

Replace this placeholder input with an appropriate domain-specific example for your agent.

In [0]:
# TODO: replace this placeholder input example with different questions to test
for event in agent.stream({"messages": [{"role": "user", "content": "In which movie did Jack climb a beanstalk?"}]}):
    print(event)

In [0]:
mlflow.models.set_model(agent)